In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

matches = spark.table("clubdata.silver.matches")
venues = spark.table("clubdata.silver.venues")

weather = (
    spark.table("clubdata.silver.weather_observations")
    .filter(
        F.col("element").isin(
            "air_temperature",
            "sum(precipitation_amount PT1H)",
            "wind_speed",
        )
    )
)

In [0]:
weather_candidates = (
    matches
    .select("match_id", "kickoff_at", "venue_id")
    .filter(F.col("venue_id").isNotNull())
    .join(weather, on="venue_id", how="inner")
    .withColumn(
        "time_distance_seconds",
        F.abs(
            F.col("observed_at").cast("long")
            - F.col("kickoff_at").cast("long")
        ),
    )
    .filter(F.col("time_distance_seconds") <= 3 * 60 * 60)
    .withColumn(
        "is_after_kickoff",
        F.when(
            F.col("observed_at") > F.col("kickoff_at"),
            1,
        ).otherwise(0),
    )
)

candidate_times = (
    weather_candidates
    .select(
        "match_id",
        "venue_id",
        "kickoff_at",
        "weather_station_id",
        "observed_at",
        "distance_to_venue_km",
        "time_distance_seconds",
        "is_after_kickoff",
    )
    .dropDuplicates()
)

weather_choice = Window.partitionBy("match_id").orderBy(
    F.asc("time_distance_seconds"),
    F.asc("is_after_kickoff"),
    F.asc("observed_at"),
    F.asc("distance_to_venue_km"),
    F.asc("weather_station_id"),
)

selected_times = (
    candidate_times
    .withColumn(
        "weather_rank",
        F.row_number().over(weather_choice),
    )
    .filter(F.col("weather_rank") == 1)
)

weather_snapshots = (
    selected_times.alias("selected")
    .join(
        weather.alias("weather"),
        on=(
            (F.col("selected.venue_id") == F.col("weather.venue_id"))
            & (
                F.col("selected.weather_station_id")
                == F.col("weather.weather_station_id")
            )
            & (
                F.col("selected.observed_at")
                == F.col("weather.observed_at")
            )
        ),
        how="inner",
    )
    .select(
        F.col("selected.match_id").alias("match_id"),
        F.col("selected.observed_at").alias("weather_observed_at"),
        F.col("weather.element").alias("element"),
        F.col("weather.value").alias("value"),
    )
    .groupBy("match_id", "weather_observed_at")
    .agg(
        F.max(
            F.when(F.col("element") == "air_temperature", F.col("value"))
        ).alias("temperature_c"),
        F.max(
            F.when(
                F.col("element") == "sum(precipitation_amount PT1H)",
                F.col("value"),
            )
        ).alias("precipitation_mm"),
        F.max(
            F.when(F.col("element") == "wind_speed", F.col("value"))
        ).alias("wind_speed_ms"),
    )
)

In [0]:
focus_team_id = 293

scored = (
    F.when(
        F.col("home_team_id") == focus_team_id,
        F.col("home_score"),
    )
    .when(
        F.col("away_team_id") == focus_team_id,
        F.col("away_score"),
    )
)

conceded = (
    F.when(
        F.col("home_team_id") == focus_team_id,
        F.col("away_score"),
    )
    .when(
        F.col("away_team_id") == focus_team_id,
        F.col("home_score"),
    )
)

finished = (
    F.lower(F.col("status")).isin("complete", "finished")
    & scored.isNotNull()
    & conceded.isNotNull()
)

matches_with_result = (
    matches
    .withColumn(
        "result",
        F.when(
            finished & (scored > conceded),
            "win",
        )
        .when(
            finished & (scored < conceded),
            "loss",
        )
        .when(
            finished & (scored == conceded),
            "draw",
        )
        .otherwise(F.lit(None).cast("string")),
    )
)

In [0]:
match_insights = (
    matches_with_result
    .join(
        venues.select(
            "venue_id",
            "stadium_name",
            "country",
            "latitude",
            "longitude",
        ),
        on="venue_id",
        how="left",
    )
    .join(
        weather_snapshots,
        on="match_id",
        how="left",
    )
    .select(
        "match_id",
        "kickoff_at",
        "competition",
        "season",
        "home_team_name",
        "away_team_name",
        "home_score",
        "away_score",
        "result",
        "venue_id",
        "stadium_name",
        "country",
        "latitude",
        "longitude",
        "weather_observed_at",
        "temperature_c",
        "precipitation_mm",
        "wind_speed_ms",
    )
)

In [0]:
(
    match_insights.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("clubdata.gold.match_insights")
)

print("Opprettet clubdata.gold.match_insights")

In [0]:
%sql

SELECT
    COUNT(*) AS gold_rows,
    COUNT(DISTINCT match_id) AS distinct_match_ids,
    COUNT_IF(
        match_id IS NULL OR kickoff_at IS NULL
    ) AS invalid_required_rows,
    COUNT_IF(
        latitude IS NOT NULL AND longitude IS NOT NULL
    ) AS matches_with_coordinates,
    COUNT_IF(weather_observed_at IS NOT NULL) AS matches_with_weather,
    ROUND(
        MAX(
            ABS(
                UNIX_TIMESTAMP(weather_observed_at)
                - UNIX_TIMESTAMP(kickoff_at)
            )
        ) / 60.0,
        1
    ) AS max_weather_offset_minutes
FROM clubdata.gold.match_insights;

In [0]:
%sql

SELECT
    kickoff_at,
    home_team_name,
    away_team_name,
    result,
    stadium_name,
    temperature_c,
    precipitation_mm,
    wind_speed_ms
FROM clubdata.gold.match_insights
ORDER BY kickoff_at DESC;